# Smart Bioinformatics Application: DNA Pattern Identification & Mutation Detection
### University Project Demonstration & Algorithm Benchmarking Notebook

**Authoritative Scientific References**:
- **NCBI Nucleotide (RefSeq)**: `NM_000518.5`, `NM_007294.4`, `NM_000492.4`
- **NCBI ClinVar**: `VCV000015147.16`, `VCV000017662.66`, `VCV000007105.15`
- **dbSNP**: `rs334`, `rs80357906`, `rs113993960`
- **Ensembl Genome Browser**: `ENSG00000244734`, `ENSG00000012048`, `ENSG00000001626`

**Evaluated Human Genes**:
1. **HBB** (Hemoglobin Subunit Beta) — Sickle Cell Anemia (`ClinVar: 15147` / `rs334` / `NM_000518.5:c.20A>T`)
2. **BRCA1** (DNA Repair Associated) — HBOC Frameshift (`ClinVar: 17662` / `rs80357906` / `NM_007294.4:c.68_69delAG`)
3. **CFTR** (CF Transmembrane Conductance Regulator) — Cystic Fibrosis (`ClinVar: 7105` / `rs113993960` / `NM_000492.4:c.1521_1523delCTT`)

> **Important Scope & Clinical Disclaimer**:
> This notebook demonstrates deterministic string matching algorithms (KMP, Boyer-Moore, Suffix Array) and Needleman-Wunsch dynamic programming alignment with HGVS 3'-rule indel normalization. These computational string matching techniques are intended for educational demonstration and algorithm verification. This 3-gene reference benchmark does NOT constitute a sufficient dataset for training or deploying clinical machine learning diagnostic models.

In [ ]:
# 1. SETUP & VERIFIED BENCHMARK DATASET INGESTION
import json
import os
from typing import List, Dict, Any, Tuple

# Authoritative reference dataset with explicit coordinate offsets
BENCHMARK_GENES = {
    "HBB": {
        "gene_symbol": "HBB",
        "name": "Hemoglobin Subunit Beta",
        "organism": "Homo sapiens",
        "ref_accession": "NM_000518.5",
        "protein_accession": "NP_000509.1",
        "region": "Full Coding Sequence (CDS c.1..444)",
        "coord_offset": 0,
        "ref_seq": (
            "ATGGTGCACCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGT"
            "GAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCT"
            "GATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCAC"
            "CTGGACAACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGATCCTGAGAACTTC"
            "AGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCC"
            "TATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTGGCCCACAAGTATCACTAA"
        ),
        "alt_name": "Sickle Cell Anemia Allele (HbS)",
        "clinvar_id": "15147",
        "clinvar_vcv": "VCV000015147.16",
        "dbsnp_rsid": "rs334",
        "hgvs_coding": "NM_000518.5:c.20A>T",
        "hgvs_protein": "NP_000509.1:p.Glu7Val",
        "clinical_condition": "Sickle Cell Disease / Sickle Cell Anemia",
        "alt_seq": (
            "ATGGTGCACCTGACTCCTGTGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGT"
            "GAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCT"
            "GATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCAC"
            "CTGGACAACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGATCCTGAGAACTTC"
            "AGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCC"
            "TATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTGGCCCACAAGTATCACTAA"
        ),
        "target_motif": "CCTGTGGAG"
    },
    "BRCA1": {
        "gene_symbol": "BRCA1",
        "name": "BRCA1 DNA Repair Associated",
        "organism": "Homo sapiens",
        "ref_accession": "NM_007294.4",
        "protein_accession": "NP_009225.1",
        "region": "Exon 2 (Coding Exon 1, c.1..175)",
        "coord_offset": 0,
        "ref_seq": (
            "ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAAATCTTAGAGACTGATTTTC"
            "AGGGTAAATGATGTGGTGAGAGCTTCAGTCACAGATTCAGTAAAATATACTCAGACATTTGGAGTATGCAAACAGCTAT"
            "ATACTAACCAGGAGAAG"
        ),
        "alt_name": "BRCA1 Frameshift Mutation (185delAG)",
        "clinvar_id": "17662",
        "clinvar_vcv": "VCV000017662.66",
        "dbsnp_rsid": "rs80357906",
        "hgvs_coding": "NM_007294.4:c.68_69delAG",
        "hgvs_protein": "NP_009225.1:p.Glu23ValfsTer17",
        "clinical_condition": "Hereditary Breast and Ovarian Cancer Syndrome (HBOC)",
        "alt_seq": (
            "ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAAATCTTAGACTGATTTTC"
            "AGGGTAAATGATGTGGTGAGAGCTTCAGTCACAGATTCAGTAAAATATACTCAGACATTTGGAGTATGCAAACAGCTAT"
            "ATACTAACCAGGAGAAG"
        ),
        "target_motif": "TCTTCGCGTT"
    },
    "CFTR": {
        "gene_symbol": "CFTR",
        "name": "CF Transmembrane Conductance Regulator",
        "organism": "Homo sapiens",
        "ref_accession": "NM_000492.4",
        "protein_accession": "NP_000483.3",
        "region": "Exon 11 Diagnostic Segment (c.1449..1554)",
        "coord_offset": 1448,
        "ref_seq": (
            "GCACATTGGAAGAATTTCATTCTGTTCTCAGTTTTCCTGGATTATGCCTGGCACCATTAAAGAAAATATCATCTTTGGT"
            "GTTTCCTATGATGAATATAGATACAGA"
        ),
        "alt_name": "CFTR Delta-F508 In-Frame Deletion",
        "clinvar_id": "7105",
        "clinvar_vcv": "VCV000007105.15",
        "dbsnp_rsid": "rs113993960",
        "hgvs_coding": "NM_000492.4:c.1521_1523delCTT",
        "hgvs_protein": "NP_000483.3:p.Phe508del",
        "clinical_condition": "Cystic Fibrosis (CF)",
        "alt_seq": (
            "GCACATTGGAAGAATTTCATTCTGTTCTCAGTTTTCCTGGATTATGCCTGGCACCATTAAAGAAAATATCATTGGTGTT"
            "TCCTATGATGAATATAGATACAGA"
        ),
        "target_motif": "ATCATTGGTG"
    }
}

print("=== VERIFIED GENOMICS DATASET INVENTORY ===")
for sym, g in BENCHMARK_GENES.items():
    print(f"{sym}: {g['name']}")
    print(f"  Region: {g['region']}")
    print(f"  Ref Accession: {g['ref_accession']} ({len(g['ref_seq'])} bp)")
    print(f"  Verified Variant: {g['alt_name']} | ClinVar: {g['clinvar_id']} | rsID: {g['dbsnp_rsid']}")
    print(f"  Expected Canonical HGVS: {g['hgvs_coding']}")


## 2. Knuth-Morris-Pratt (KMP) Exact Pattern Search
The KMP algorithm constructs a $\pi$ (LPS) table to track the longest proper prefix matching a suffix, skipping unnecessary comparisons upon mismatch.
- Time Complexity: **$O(N + M)$**

In [ ]:
def compute_lps_table(pattern: str) -> List[int]:
    m = len(pattern)
    lps = [0] * m
    length = 0
    i = 1
    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1
    return lps

def kmp_search(text: str, pattern: str) -> Dict[str, Any]:
    n, m = len(text), len(pattern)
    if m == 0 or n == 0 or m > n:
        return {"matches": [], "count": 0, "comparisons": 0, "lps": []}
    lps = compute_lps_table(pattern)
    occurrences = []
    i, j, comparisons = 0, 0, 0
    while i < n:
        comparisons += 1
        if text[i] == pattern[j]:
            i += 1
            j += 1
        if j == m:
            occurrences.append(i - j)
            j = lps[j - 1]
        elif i < n and text[i] != pattern[j]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return {"matches": occurrences, "count": len(occurrences), "comparisons": comparisons, "lps": lps}

# Test KMP on HBB Sickle Motif
hbb_alt = BENCHMARK_GENES["HBB"]["alt_seq"]
motif = BENCHMARK_GENES["HBB"]["target_motif"]
res_kmp = kmp_search(hbb_alt, motif)
print(f"KMP search for '{motif}' in HBB mutant sequence:")
print(f"  LPS Table: {res_kmp['lps']}")
print(f"  Matches at 1-based coordinates: {[m + 1 for m in res_kmp['matches']]}")
print(f"  Comparisons: {res_kmp['comparisons']}")


## 3. Boyer-Moore (BM) Motif Search
Scans the pattern right-to-left and uses the Bad Character jump rule to skip non-matching segments.
- Average Time Complexity: **$O(N / M)$** sub-linear performance

In [ ]:
def build_bad_char_table(pattern: str) -> Dict[str, int]:
    return {pattern[i]: i for i in range(len(pattern))}

def boyer_moore_search(text: str, pattern: str) -> Dict[str, Any]:
    n, m = len(text), len(pattern)
    if m == 0 or n == 0 or m > n:
        return {"matches": [], "count": 0, "comparisons": 0, "bad_char_table": {}}
    bad_char = build_bad_char_table(pattern)
    occurrences = []
    comparisons = 0
    s = 0
    while s <= (n - m):
        j = m - 1
        while j >= 0:
            comparisons += 1
            if pattern[j] == text[s + j]:
                j -= 1
            else:
                break
        if j < 0:
            occurrences.append(s)
            s += (m - bad_char.get(text[s + m], -1)) if (s + m < n) else 1
        else:
            mismatched = text[s + j]
            last_occ = bad_char.get(mismatched, -1)
            s += max(1, j - last_occ)
    return {"matches": occurrences, "count": len(occurrences), "comparisons": comparisons, "bad_char_table": bad_char}

res_bm = boyer_moore_search(hbb_alt, motif)
print(f"Boyer-Moore search for '{motif}':")
print(f"  Bad Character Table: {res_bm['bad_char_table']}")
print(f"  Matches at 1-based coordinates: {[m + 1 for m in res_bm['matches']]}")
print(f"  Comparisons: {res_bm['comparisons']} vs KMP ({res_kmp['comparisons']})")


## 4. Suffix Array & Kasai LCP Array
Computes suffix ordering and the Longest Common Prefix array to discover repeats, duplications, and the Longest Repeated Substring (LRS).

In [ ]:
def build_suffix_array(text: str) -> List[int]:
    suffixes = [(text[i:], i) for i in range(len(text))]
    suffixes.sort(key=lambda x: x[0])
    return [s[1] for s in suffixes]

def kasai_lcp(text: str, sa: List[int]) -> List[int]:
    n = len(text)
    rank = [0] * n
    lcp = [0] * n
    for i in range(n):
        rank[sa[i]] = i
    k = 0
    for i in range(n):
        if rank[i] == 0:
            k = 0
            continue
        j = sa[rank[i] - 1]
        while i + k < n and j + k < n and text[i + k] == text[j + k]:
            k += 1
        lcp[rank[i]] = k
        if k > 0:
            k -= 1
    return lcp

def find_repeats(seq: str, min_len: int = 5) -> List[Dict[str, Any]]:
    sa = build_suffix_array(seq)
    lcp = kasai_lcp(seq, sa)
    repeats, seen = [], set()
    for i in range(1, len(lcp)):
        length = lcp[i]
        if length >= min_len:
            sub = seq[sa[i]:sa[i] + length]
            if sub not in seen:
                seen.add(sub)
                repeats.append({"motif": sub, "length": length, "positions": sorted([sa[i - 1] + 1, sa[i] + 1])})
    repeats.sort(key=lambda x: x["length"], reverse=True)
    return repeats

hbb_ref = BENCHMARK_GENES["HBB"]["ref_seq"]
repeats = find_repeats(hbb_ref, min_len=6)
print(f"Top repeated motifs in HBB Reference Sequence:")
for r in repeats[:5]:
    print(f"  - Motif '{r['motif']}' ({r['length']} bp) at positions {r['positions']}")


## 5. Needleman-Wunsch Dynamic Programming Alignment & HGVS Normalization
Implements global pairwise alignment, contiguous indel grouping, coordinate offset mapping, and HGVS 3'-rule right-shift normalization.

In [ ]:
def find_equivalent_deletions(
    unaligned_ref: str, unaligned_alt: str, raw_start: int, raw_end: int, coord_offset: int = 0, transcript: str = ""
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    k = raw_end - raw_start + 1
    n = len(unaligned_ref)
    prefix = f"{transcript}:" if transcript else ""
    s, e = raw_start, raw_end
    while s > 1:
        if unaligned_ref[s - 2] == unaligned_ref[e - 1]:
            s -= 1
            e -= 1
        else:
            break
    equivs = []
    while e <= n:
        if unaligned_ref[:s - 1] + unaligned_ref[e:] == unaligned_alt:
            cur_del = unaligned_ref[s - 1:e]
            cds_s = s + coord_offset
            cds_e = e + coord_offset
            hgvs = f"{prefix}c.{cds_s}_{cds_e}del{cur_del}" if k > 1 else f"{prefix}c.{cds_s}del{cur_del}"
            equivs.append({
                "local_start": s, "local_end": e, "deleted_sequence": cur_del,
                "cds_start": cds_s, "cds_end": cds_e, "hgvs_coding": hgvs
            })
        if e < n and unaligned_ref[s - 1] == unaligned_ref[e]:
            s += 1
            e += 1
        else:
            break
    if not equivs:
        cds_s, cds_e = raw_start + coord_offset, raw_end + coord_offset
        cur_del = unaligned_ref[raw_start - 1:raw_end]
        hgvs = f"{prefix}c.{cds_s}_{cds_e}del{cur_del}" if k > 1 else f"{prefix}c.{cds_s}del{cur_del}"
        item = {"local_start": raw_start, "local_end": raw_end, "deleted_sequence": cur_del, "cds_start": cds_s, "cds_end": cds_e, "hgvs_coding": hgvs}
        return item, [item]
    raw_del = unaligned_ref[raw_start - 1:raw_end]
    matching = [x for x in equivs if x["deleted_sequence"] == raw_del]
    canonical = matching[-1] if len(matching) > 1 else equivs[-1]
    return canonical, equivs

def needleman_wunsch_align(
    ref: str, alt: str, match: int = 2, mismatch: int = -1, gap: int = -2,
    coord_offset: int = 0, transcript: str = ""
) -> Dict[str, Any]:
    n, m = len(ref), len(alt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1): dp[i][0] = dp[i - 1][0] + gap
    for j in range(1, m + 1): dp[0][j] = dp[0][j - 1] + gap
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s_match = dp[i - 1][j - 1] + (match if ref[i - 1] == alt[j - 1] else mismatch)
            dp[i][j] = max(s_match, dp[i - 1][j] + gap, dp[i][j - 1] + gap)
    aligned_ref, aligned_alt = [], []
    i, j = n, m
    while i > 0 or j > 0:
        score = dp[i][j]
        if i > 0 and j > 0 and score == dp[i - 1][j - 1] + (match if ref[i - 1] == alt[j - 1] else mismatch):
            aligned_ref.append(ref[i - 1]); aligned_alt.append(alt[j - 1]); i -= 1; j -= 1
        elif i > 0 and score == dp[i - 1][j] + gap:
            aligned_ref.append(ref[i - 1]); aligned_alt.append('-'); i -= 1
        else:
            aligned_ref.append('-'); aligned_alt.append(alt[j - 1]); j -= 1
    aligned_ref = ''.join(reversed(aligned_ref))
    aligned_alt = ''.join(reversed(aligned_alt))

    prefix = f"{transcript}:" if transcript else ""
    mutations = []
    ref_pos, alt_pos = 0, 0
    idx, total_cols = 0, len(aligned_ref)
    matches, mismatches, gaps = 0, 0, 0

    while idx < total_cols:
        r, a = aligned_ref[idx], aligned_alt[idx]
        if r != '-': ref_pos += 1
        if a != '-': alt_pos += 1
        if r == a:
            matches += 1; idx += 1
        elif r != '-' and a != '-':
            mismatches += 1
            sub_type = 'TRANSITION' if (r, a) in [('A','G'),('G','A'),('C','T'),('T','C')] else 'TRANSVERSION'
            cds_pos = ref_pos + coord_offset
            mutations.append({
                'type': 'SUBSTITUTION', 'pos': ref_pos, 'cds_pos': cds_pos, 'ref': r, 'alt': a,
                'hgvs': f'{prefix}c.{cds_pos}{r}>{a}', 'class': sub_type, 'equivs': []
            })
            idx += 1
        elif r != '-' and a == '-':
            del_start = ref_pos; del_bases = [r]; gaps += 1; k = idx + 1
            while k < total_cols and aligned_ref[k] != '-' and aligned_alt[k] == '-':
                ref_pos += 1; del_bases.append(aligned_ref[k]); gaps += 1; k += 1
            del_end = ref_pos
            canon, eq_list = find_equivalent_deletions(ref, alt, del_start, del_end, coord_offset, transcript)
            mutations.append({
                'type': 'DELETION', 'pos': canon['local_start'], 'cds_pos': canon['cds_start'],
                'ref': canon['deleted_sequence'], 'alt': '-', 'hgvs': canon['hgvs_coding'],
                'class': f"{len(canon['deleted_sequence'])}bp deletion", 'equivs': eq_list
            })
            idx = k
        else:
            ins_start = ref_pos; ins_bases = [a]; gaps += 1; k = idx + 1
            while k < total_cols and aligned_ref[k] == '-' and aligned_alt[k] != '-':
                alt_pos += 1; ins_bases.append(aligned_alt[k]); gaps += 1; k += 1
            ins_seq = ''.join(ins_bases)
            cds_ins = ins_start + coord_offset
            mutations.append({
                'type': 'INSERTION', 'pos': ins_start, 'cds_pos': cds_ins, 'ref': '-', 'alt': ins_seq,
                'hgvs': f'{prefix}c.{cds_ins}_{cds_ins+1}ins{ins_seq}', 'class': f'{len(ins_seq)}bp insertion', 'equivs': []
            })
            idx = k
    return {
        'score': dp[n][m], 'identity': round((matches / total_cols) * 100, 2),
        'matches': matches, 'mismatches': mismatches, 'gaps': gaps,
        'aligned_ref': aligned_ref, 'aligned_alt': aligned_alt, 'mutations': mutations
    }


## 6. Comprehensive Verification Pipeline & Biological Equivalence Validation
Execute the complete pipeline on all three human genes to verify exact sequence changes, coordinate conversion, and HGVS nomenclature reporting.

In [ ]:
for gene_sym, g in BENCHMARK_GENES.items():
    print("=" * 80)
    print(f"GENE: {gene_sym} ({g['name']})")
    print(f"Reference: {g['ref_accession']} | Region: {g['region']} | Offset: +{g['coord_offset']} bp")
    print(f"Target Variant: {g['alt_name']} | ClinVar: {g['clinvar_id']} ({g['clinvar_vcv']})")
    print(f"Expected HGVS: {g['hgvs_coding']}")
    print("-" * 80)

    aln = needleman_wunsch_align(
        g['ref_seq'], g['alt_seq'],
        coord_offset=g['coord_offset'],
        transcript=g['ref_accession']
    )
    print(f"Alignment Score: {aln['score']} | Sequence Identity: {aln['identity']}%")
    print(f"Matches: {aln['matches']} | Mismatches: {aln['mismatches']} | Gaps: {aln['gaps']}")
    print(f"Detected Mutations ({len(aln['mutations'])}):")
    for m in aln['mutations']:
        print(f"  ==> Detected HGVS: {m['hgvs']} [{m['type']} | {m['class']}]")
        print(f"      cDNA Coordinate: c.{m['cds_pos']} (Local position: {m['pos']})")
        if m['equivs'] and len(m['equivs']) > 1:
            print("      Equivalent deletion windows (HGVS 3'-normalization applied):")
            for eq in m['equivs']:
                print(f"        - {eq['hgvs_coding']} (local {eq['local_start']}..{eq['local_end']} del{eq['deleted_sequence']})")

    p = aln['mutations'][0]['pos'] if aln['mutations'] else 10
    w_start = max(0, p - 12)
    w_end = min(len(aln['aligned_ref']), p + 12)
    r_sub = aln['aligned_ref'][w_start:w_end]
    a_sub = aln['aligned_alt'][w_start:w_end]
    bar = ''.join('|' if r_sub[k] == a_sub[k] else ('.' if '-' not in (r_sub[k], a_sub[k]) else ' ') for k in range(len(r_sub)))
    print(f"\n  Alignment Slice (local pos {w_start+1}..{w_end}):")
    print(f"  Ref: {r_sub}")
    print(f"       {bar}")
    print(f"  Alt: {a_sub}\n")
